# Heterogeneous SHD SNN with ANN Weight modulation

This is an SHD version of the `Weight_mod/weight_modulation.py` experiment (running on software, not BSS2)

Procedure:
1. Compress 700 cochlea channels to 70 channels in groups of 10, (like `NM_EEG/snn_allinone_EEG.py` compression)
3. Run 100-timestep recurrent SNN with 70 inputs, 128 hidden neurons, 20 outputs
4. Use heterogeneous hidden/readout neuron parameters inside surrogate-gradient SNN.
5. Every `ann_interval` timesteps: give ANN modulator recent input spike sums, recent hidden spike sums, current output membrane, and release-site weight statistics
6. ANN outputs release amplitudes/spreads for `w1`, `v1`, and `w2` (like `Weight_mod/weight_modulation.py`) which change the SNN weights during runtime (100-timestep inference window?)

This is off-chip training. Next do the same with PPU-weight editing during `pynn.run`. See older notebooks for writing PPU code, and these ideas for how to get the values:
    - `input_spike_sum`: reconstruct from known input spike source schedule or accumulated by a PPU-side input-event counter(?)
    - `hidden_spike_sum`: needs access to hidden spike observables or a routed counter population
    - `output_mem`: needs neuron/CADC/MADC-compatible read path, or proxy output-spike/current state if membrane readout conflicts with plasticity
    - `weight stats`: compute directly on PPU, because PPU owns the synapse view it is modifying

TODO NEXT: hardware probe for reading/approximating `hidden_spike_sum` and `output_mem` inside a plasticity rule.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import h5py
from urllib.request import urlretrieve
import shutil
import gzip
import os
import hashlib
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import quantities as pq

from dlens_vx_v3 import hal, halco
import pynn_brainscales.brainscales2 as pynn
from _static.common.helpers import setup_hardware_client, save_nightly_calibration
from _static.tutorial.hxsnn_intro_plots import plot_compare_traces

setup_hardware_client()

from hxtorch import snn as hxsnn
import hxtorch
import pygrenade_vx as grenade


hello
INFO  10:00:47,467  demo_helpers Connection to ['hxcube6fpga3chip66_1'] established


In [2]:
import math
from datetime import datetime
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Tuple

SHD_URLS = {
    "shd_train.h5": (
        "https://compneuro.net/datasets/shd_train.h5.gz",
        "d47c9825dee33347913e8ce0f2be08b0",
    ),
    "shd_test.h5": (
        "https://compneuro.net/datasets/shd_test.h5.gz",
        "3062a80ec0c5719404d5b02e166543b1",
    ),
}


def _md5(path, chunk_size=1 << 20):
    digest = hashlib.md5()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_shd_download(root="SHD_BSS2/data"):
    """Download, verify, and decompress SHD files if missing."""
    root = Path(root).expanduser()
    root.mkdir(parents=True, exist_ok=True)
    for filename, (url, expected_md5) in SHD_URLS.items():
        out_path = root / filename
        gz_path = root / f"{filename}.gz"
        if out_path.exists():
            print(f"found {out_path}")
            continue
        if not gz_path.exists():
            print(f"downloading {url} -> {gz_path}")
            urlretrieve(url, gz_path)
        digest = _md5(gz_path)
        if digest != expected_md5:
            raise RuntimeError(
                f"MD5 mismatch for {gz_path}: got {digest}, expected {expected_md5}. "
                "Delete the .gz file and rerun the cell."
            )
        print(f"decompressing {gz_path} -> {out_path}")
        with gzip.open(gz_path, "rb") as src_file, out_path.open("wb") as dst_file:
            shutil.copyfileobj(src_file, dst_file)
    return root


def open_shd_h5(root="SHD_BSS2/data"):
    root = Path(root).expanduser()
    ensure_shd_download(root)
    train_path = root / "shd_train.h5"
    test_path = root / "shd_test.h5"
    return h5py.File(train_path, "r"), h5py.File(test_path, "r")


def stratified_subset(labels, limit=None, seed=0):
    labels = np.asarray(labels, dtype=np.int64)
    if limit is None or int(limit) <= 0 or int(limit) >= len(labels):
        return np.arange(len(labels), dtype=np.int64)
    rng = np.random.default_rng(seed)
    classes = np.unique(labels)
    per_class = max(1, int(math.ceil(int(limit) / len(classes))))
    chosen = []
    for cls in classes:
        idx = np.where(labels == cls)[0]
        rng.shuffle(idx)
        chosen.extend(idx[:per_class].tolist())
    chosen = np.asarray(chosen[:int(limit)], dtype=np.int64)
    rng.shuffle(chosen)
    return chosen


def compressed_shd_sparse_batches(
    h5_file,
    indices,
    batch_size=32,
    nb_steps=100,
    nb_inputs_raw=700,
    nb_inputs=70,
    max_time=1.0,
    shuffle=True,
    seed=0,
):
    """Yield dense [batch, time, 70] SHD spikes after 700->70 channel grouping."""
    indices = np.asarray(indices, dtype=np.int64).copy()
    rng = np.random.default_rng(seed)
    if shuffle:
        rng.shuffle(indices)
    labels_all = np.asarray(h5_file["labels"], dtype=np.int64)
    times_ds = h5_file["spikes/times"]
    units_ds = h5_file["spikes/units"]
    compress_factor = max(1, int(math.ceil(float(nb_inputs_raw) / float(nb_inputs))))
    time_bins = np.linspace(0.0, float(max_time), num=int(nb_steps), endpoint=False)

    for start in range(0, len(indices) - len(indices) % int(batch_size), int(batch_size)):
        batch_idx = indices[start:start + int(batch_size)]
        coo = [[], [], []]
        for bc, sample_idx in enumerate(batch_idx):
            times = np.asarray(times_ds[int(sample_idx)], dtype=np.float32)
            units = np.asarray(units_ds[int(sample_idx)], dtype=np.int64)
            valid = (times >= 0.0) & (times <= float(max_time)) & (units >= 0) & (units < int(nb_inputs_raw))
            times = times[valid]
            units = units[valid]
            if len(times) == 0:
                continue
            tbin = np.searchsorted(time_bins, times, side="right") - 1
            tbin = np.clip(tbin, 0, int(nb_steps) - 1)
            cbin = np.floor_divide(units, compress_factor)
            cbin = np.clip(cbin, 0, int(nb_inputs) - 1)
            coo[0].extend([bc] * len(tbin))
            coo[1].extend(tbin.tolist())
            coo[2].extend(cbin.tolist())
        if coo[0]:
            idx = torch.tensor(coo, dtype=torch.long)
            val = torch.ones(len(coo[0]), dtype=torch.float32)
            x = torch.sparse_coo_tensor(
                idx,
                val,
                size=(len(batch_idx), int(nb_steps), int(nb_inputs)),
                dtype=torch.float32,
            ).coalesce().to_dense().clamp_max(1.0)
        else:
            x = torch.zeros((len(batch_idx), int(nb_steps), int(nb_inputs)), dtype=torch.float32)
        y = torch.tensor(labels_all[batch_idx], dtype=torch.long)
        yield x, y


In [3]:
class SHDSurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output / (SHDSurrGradSpike.scale * x.abs() + 1.0).pow(2)


shd_spike_fn = SHDSurrGradSpike.apply


def shd_nearest_rect_grid(n: int) -> Tuple[int, int]:
    rows = max(1, int(math.floor(math.sqrt(max(1, n)))))
    cols = max(1, int(math.ceil(max(1, n) / rows)))
    while rows * cols < n:
        rows += 1
        cols = max(1, int(math.ceil(n / rows)))
    return rows, cols


def shd_positions(n: int, grid_dim="2d", device=None):
    if grid_dim == "1d":
        return torch.arange(n, device=device, dtype=torch.float32).unsqueeze(1)
    rows, cols = shd_nearest_rect_grid(n)
    idx = torch.arange(n, device=device, dtype=torch.float32)
    return torch.stack([torch.floor(idx / cols), idx.remainder(cols)], dim=1)


def shd_max_grid_distance(n: int, grid_dim="2d", device=None):
    pos = shd_positions(n, grid_dim, device=device)
    if pos.size(0) <= 1:
        return torch.tensor(1.0, device=device)
    return torch.cdist(pos, pos).max().clamp_min(1.0)


def shd_release_site_indices(n: int, count: int, grid_dim="2d"):
    n = int(n)
    count = max(0, min(int(count), n))
    if count == 0:
        return torch.empty(0, dtype=torch.long)
    if count == n:
        return torch.arange(n, dtype=torch.long)
    return torch.linspace(0, n - 1, count).round().long().unique()


def shd_pairwise_dist_to_sites(n: int, sites: torch.Tensor, grid_dim="2d"):
    pos = shd_positions(n, grid_dim, device=sites.device)
    site_pos = pos[sites.long()]
    return torch.cdist(site_pos, pos)


def shd_fixed_summary_kernels(n: int, sites: torch.Tensor, grid_dim="2d", width=2.0, mode="uniform"):
    if sites.numel() == 0:
        return torch.empty((0, n), dtype=torch.float32)
    dist = shd_pairwise_dist_to_sites(n, sites, grid_dim)
    if mode == "normal":
        weights = torch.exp(-0.5 * (dist / max(float(width), 1e-6)).pow(2))
    else:
        weights = (dist <= max(float(width), 0.0)).float()
    return weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-6)


def shd_spread_kernels(n: int, sites: torch.Tensor, spread01: torch.Tensor, grid_dim="2d", mode="normal"):
    if sites.numel() == 0:
        return torch.empty((spread01.size(0), 0, n), device=spread01.device, dtype=spread01.dtype)
    dist = shd_pairwise_dist_to_sites(n, sites.to(spread01.device), grid_dim).to(spread01.dtype)
    max_dist = shd_max_grid_distance(n, grid_dim, device=spread01.device).to(spread01.dtype)
    spread = spread01.clamp(0.0, 1.0).unsqueeze(2)
    if mode == "normal":
        sigma = (spread * max_dist).clamp_min(1e-6)
        weights = torch.exp(-0.5 * (dist.unsqueeze(0) / sigma).pow(2))
        point = (dist.unsqueeze(0) <= 0.5).to(weights.dtype)
        return torch.where(spread <= 1e-6, point, weights)
    radius = spread * max_dist
    return (dist.unsqueeze(0) <= radius.clamp_min(0.5)).to(spread01.dtype)


def shd_incoming_weight_stats(matrix: torch.Tensor, target_kernels: torch.Tensor):
    if target_kernels.numel() == 0:
        return matrix.new_zeros((matrix.size(0), 0))
    kernels = target_kernels.to(device=matrix.device, dtype=matrix.dtype)
    denom = float(matrix.size(1))
    mean = torch.einsum("bst,rt->br", matrix, kernels) / max(1.0, denom)
    abs_mean = torch.einsum("bst,rt->br", matrix.abs(), kernels) / max(1.0, denom)
    second = torch.einsum("bst,rt->br", matrix * matrix, kernels) / max(1.0, denom)
    std = torch.sqrt(torch.clamp(second - mean * mean, min=0.0))
    return torch.cat([mean, abs_mean, std], dim=1)


@dataclass
class SHDReleaseConfig:
    nb_inputs: int = 70
    nb_inputs_raw: int = 700
    nb_hidden: int = 128
    nb_outputs: int = 20
    nb_steps: int = 100
    max_time: float = 1.0
    time_step: float = 1e-3
    tau_syn: float = 10e-3
    tau_mem: float = 20e-3
    ann_interval: int = 3
    release_hidden: int = 32
    release_output: int = 8
    grid_dim: str = "2d"
    spread_mode: str = "normal"
    summary_width: float = 2.0
    summary_mode: str = "uniform"
    delta_scale: float = 0.05
    hidden_sizes: Tuple[int, ...] = (256,)
    alpha_range: Tuple[float, float] = (1.0 / math.e, 0.995)
    beta_range: Tuple[float, float] = (1.0 / math.e, 0.995)
    thr_range: Tuple[float, float] = (0.5, 1.5)
    reset_range: Tuple[float, float] = (-0.5, 0.5)
    rest_range: Tuple[float, float] = (-0.5, 0.5)


class SHDReleaseSiteMLP(nn.Module):
    """Weight_mod-style ANN: activity + weight stats -> release amplitudes/spreads."""

    def __init__(self, cfg: SHDReleaseConfig):
        super().__init__()
        self.cfg = cfg
        self.hidden_sites = shd_release_site_indices(cfg.nb_hidden, cfg.release_hidden, cfg.grid_dim)
        self.output_sites = shd_release_site_indices(cfg.nb_outputs, cfg.release_output, cfg.grid_dim)
        self.hidden_summary = shd_fixed_summary_kernels(
            cfg.nb_hidden, self.hidden_sites, cfg.grid_dim, cfg.summary_width, cfg.summary_mode
        )
        self.output_summary = shd_fixed_summary_kernels(
            cfg.nb_outputs, self.output_sites, cfg.grid_dim, cfg.summary_width, cfg.summary_mode
        )
        self.hidden_release_count = int(self.hidden_sites.numel())
        self.output_release_count = int(self.output_sites.numel())
        activity_dim = cfg.nb_inputs + cfg.nb_hidden + cfg.nb_outputs
        summary_dim = 3 * (2 * self.hidden_release_count + self.output_release_count)
        self.input_dim = activity_dim + summary_dim
        self.output_dim = 3 * self.hidden_release_count + 2 * self.output_release_count
        layers: List[nn.Module] = []
        prev = self.input_dim
        for width in cfg.hidden_sizes:
            layers += [nn.Linear(prev, int(width)), nn.ReLU()]
            prev = int(width)
        layers.append(nn.Linear(prev, self.output_dim))
        self.net = nn.Sequential(*layers)
        with torch.no_grad():
            for mod in self.net:
                if isinstance(mod, nn.Linear):
                    nn.init.normal_(mod.weight, mean=0.0, std=1e-4)
                    nn.init.zeros_(mod.bias)

    def build_input(self, in_flat, hid_flat, out_mem, w1_local, v1_local, w2_local):
        h_kernel = self.hidden_summary.to(device=w1_local.device, dtype=w1_local.dtype)
        o_kernel = self.output_summary.to(device=w2_local.device, dtype=w2_local.dtype)
        summaries = [
            shd_incoming_weight_stats(w1_local, h_kernel),
            shd_incoming_weight_stats(v1_local, h_kernel),
            shd_incoming_weight_stats(w2_local, o_kernel),
        ]
        return torch.cat([in_flat, hid_flat, out_mem] + summaries, dim=1)

    def forward(self, mlp_input):
        raw = self.net(mlp_input)
        rh = self.hidden_release_count
        ro = self.output_release_count
        offset = 0
        amp_w1 = torch.tanh(raw[:, offset:offset + rh]); offset += rh
        amp_v1 = torch.tanh(raw[:, offset:offset + rh]); offset += rh
        spread_h = torch.sigmoid(raw[:, offset:offset + rh]); offset += rh
        amp_w2 = torch.tanh(raw[:, offset:offset + ro]); offset += ro
        spread_o = torch.sigmoid(raw[:, offset:offset + ro])
        return {"amp_w1": amp_w1, "amp_v1": amp_v1, "spread_h": spread_h, "amp_w2": amp_w2, "spread_o": spread_o}


class SHDHeterogeneousSNN(nn.Module):
    """Trainable heterogeneous recurrent SNN state for SHD."""

    def __init__(self, cfg: SHDReleaseConfig, seed=41):
        super().__init__()
        torch.manual_seed(seed)
        self.cfg = cfg
        self.w1 = nn.Parameter(torch.empty(cfg.nb_inputs, cfg.nb_hidden).normal_(0.0, 0.02))
        self.v1 = nn.Parameter(torch.empty(cfg.nb_hidden, cfg.nb_hidden).normal_(0.0, 0.02))
        self.w2 = nn.Parameter(torch.empty(cfg.nb_hidden, cfg.nb_outputs).normal_(0.0, 0.04))
        self.raw_alpha_1 = nn.Parameter(torch.randn(cfg.nb_hidden) * 0.35)
        self.raw_beta_1 = nn.Parameter(torch.randn(cfg.nb_hidden) * 0.35)
        self.raw_thr = nn.Parameter(torch.randn(cfg.nb_hidden) * 0.35)
        self.raw_reset = nn.Parameter(torch.randn(cfg.nb_hidden) * 0.20)
        self.raw_rest = nn.Parameter(torch.randn(cfg.nb_hidden) * 0.20)
        self.raw_alpha_2 = nn.Parameter(torch.randn(cfg.nb_outputs) * 0.25)
        self.raw_beta_2 = nn.Parameter(torch.randn(cfg.nb_outputs) * 0.25)

    @staticmethod
    def _bounded(raw, bounds):
        lo, hi = bounds
        return lo + (hi - lo) * torch.sigmoid(raw)

    def neuron_params(self):
        cfg = self.cfg
        return {
            "alpha_1": self._bounded(self.raw_alpha_1, cfg.alpha_range).unsqueeze(0),
            "beta_1": self._bounded(self.raw_beta_1, cfg.beta_range).unsqueeze(0),
            "thr": self._bounded(self.raw_thr, cfg.thr_range).unsqueeze(0),
            "reset": self._bounded(self.raw_reset, cfg.reset_range).unsqueeze(0),
            "rest": self._bounded(self.raw_rest, cfg.rest_range).unsqueeze(0),
            "alpha_2": self._bounded(self.raw_alpha_2, cfg.alpha_range).unsqueeze(0),
            "beta_2": self._bounded(self.raw_beta_2, cfg.beta_range).unsqueeze(0),
        }


In [4]:
def shd_tensor_summary(tensor, include_shape=True):
    data = tensor.detach().float()
    out = {}
    if include_shape:
        out["shape"] = list(data.shape)
    if data.numel() == 0:
        out.update(mean=0.0, abs_mean=0.0, std=0.0, min=0.0, max=0.0, finite=True)
    else:
        out.update(
            mean=float(data.mean().item()),
            abs_mean=float(data.abs().mean().item()),
            std=float(data.std(unbiased=False).item()),
            min=float(data.min().item()),
            max=float(data.max().item()),
            finite=bool(torch.isfinite(data).all().item()),
        )
    return out


def shd_matrix_scale(matrix):
    max_abs = matrix.detach().abs().amax(dim=(1, 2), keepdim=True)
    std = matrix.detach().std(dim=(1, 2), keepdim=True, unbiased=False)
    return torch.maximum(max_abs, 3.0 * std).clamp_min(1e-6)


def shd_field_from_releases(amplitudes, spreads, sites, n, cfg):
    kernels = shd_spread_kernels(n, sites.to(amplitudes.device), spreads, cfg.grid_dim, cfg.spread_mode)
    if kernels.numel() == 0:
        return amplitudes.new_zeros((amplitudes.size(0), n))
    numerator = torch.sum(amplitudes.unsqueeze(2) * kernels, dim=1)
    denom = torch.sum(kernels, dim=1).clamp_min(1e-6)
    return numerator / denom


def shd_apply_release_weight_update(w1_local, v1_local, w2_local, release, modulator, cfg):
    hidden_field_w1 = shd_field_from_releases(release["amp_w1"], release["spread_h"], modulator.hidden_sites, cfg.nb_hidden, cfg)
    hidden_field_v1 = shd_field_from_releases(release["amp_v1"], release["spread_h"], modulator.hidden_sites, cfg.nb_hidden, cfg)
    output_field_w2 = shd_field_from_releases(release["amp_w2"], release["spread_o"], modulator.output_sites, cfg.nb_outputs, cfg)
    w1_next = w1_local + hidden_field_w1.unsqueeze(1) * shd_matrix_scale(w1_local) * cfg.delta_scale
    v1_next = v1_local + hidden_field_v1.unsqueeze(1) * shd_matrix_scale(v1_local) * cfg.delta_scale
    w2_next = w2_local + output_field_w2.unsqueeze(1) * shd_matrix_scale(w2_local) * cfg.delta_scale
    return w1_next, v1_next, w2_next


def run_shd_weight_release_modulated(inputs, snn, modulator, cfg, modulation_events=None):
    batch = inputs.size(0)
    params = snn.neuron_params()
    alpha_1 = params["alpha_1"].expand(batch, cfg.nb_hidden)
    beta_1 = params["beta_1"].expand(batch, cfg.nb_hidden)
    thr = params["thr"].expand(batch, cfg.nb_hidden)
    rst = params["reset"].expand(batch, cfg.nb_hidden)
    rest = params["rest"].expand(batch, cfg.nb_hidden)
    alpha_2 = params["alpha_2"].expand(batch, cfg.nb_outputs)
    beta_2 = params["beta_2"].expand(batch, cfg.nb_outputs)

    w1_local = snn.w1.unsqueeze(0).expand(batch, -1, -1).clone()
    v1_local = snn.v1.unsqueeze(0).expand(batch, -1, -1).clone()
    w2_local = snn.w2.unsqueeze(0).expand(batch, -1, -1).clone()

    syn = torch.zeros((batch, cfg.nb_hidden), device=inputs.device, dtype=inputs.dtype)
    mem = torch.zeros_like(syn)
    out_h = torch.zeros_like(syn)
    flt2 = torch.zeros((batch, cfg.nb_outputs), device=inputs.device, dtype=inputs.dtype)
    out2 = torch.zeros_like(flt2)

    spk_rec, mem_rec, out_rec = [], [], []
    k = max(1, int(cfg.ann_interval))

    for t in range(cfg.nb_steps):
        h1_t = torch.einsum("bi,bih->bh", inputs[:, t, :], w1_local)
        h1_t = h1_t + torch.einsum("bh,bhr->br", out_h, v1_local)

        mthr = mem - thr
        out_h = shd_spike_fn(mthr)
        rst_mask = (mthr > 0).float()
        syn = alpha_1 * syn + h1_t
        mem = beta_1 * (mem - rest) + rest + (1.0 - beta_1) * syn - rst_mask * (thr - rst)
        spk_rec.append(out_h)
        mem_rec.append(mem)

        h2_t = torch.einsum("bh,bho->bo", out_h, w2_local)
        flt2 = alpha_2 * flt2 + h2_t
        out2 = beta_2 * out2 + (1.0 - beta_2) * flt2
        out_rec.append(out2)

        if t % k == 0:
            start = max(0, t - k + 1)
            in_flat = inputs[:, start:t + 1, :].sum(dim=1)
            hid_flat = torch.stack(spk_rec[start:t + 1], dim=1).sum(dim=1)
            out_mem = out2
            mlp_in = modulator.build_input(in_flat, hid_flat, out_mem, w1_local, v1_local, w2_local)
            release = modulator(mlp_in)
            w1_next, v1_next, w2_next = shd_apply_release_weight_update(w1_local, v1_local, w2_local, release, modulator, cfg)
            if modulation_events is not None:
                modulation_events.append({
                    "step": int(t),
                    "window_start": int(start),
                    "window_size": int(t - start + 1),
                    "input_spike_sum": float(in_flat.detach().sum().item()),
                    "hidden_spike_sum": float(hid_flat.detach().sum().item()),
                    "output_mem_sum": float(out_mem.detach().sum().item()),
                    "amp_w1": shd_tensor_summary(release["amp_w1"], include_shape=False),
                    "amp_v1": shd_tensor_summary(release["amp_v1"], include_shape=False),
                    "amp_w2": shd_tensor_summary(release["amp_w2"], include_shape=False),
                    "spread_h": shd_tensor_summary(release["spread_h"], include_shape=False),
                    "spread_o": shd_tensor_summary(release["spread_o"], include_shape=False),
                    "delta_w1": shd_tensor_summary(w1_next - w1_local, include_shape=False),
                    "delta_v1": shd_tensor_summary(v1_next - v1_local, include_shape=False),
                    "delta_w2": shd_tensor_summary(w2_next - w2_local, include_shape=False),
                })
            w1_local, v1_local, w2_local = w1_next, v1_next, w2_next

    out_trace = torch.stack(out_rec, dim=1)
    mem_trace = torch.stack(mem_rec, dim=1)
    spk_trace = torch.stack(spk_rec, dim=1)
    logits = out_trace.max(dim=1).values
    return logits, out_trace, mem_trace, spk_trace


def shd_spike_regularization(spikes, scale=1e-3):
    if scale <= 0:
        return spikes.new_zeros(())
    mean_rate = spikes.mean(dim=1)
    high_rate = torch.clamp(mean_rate - 0.01, min=0.0).pow(2).mean()
    sample_rate = spikes.mean(dim=(1, 2))
    burst = torch.clamp(sample_rate - 0.06, min=0.0).pow(2).mean()
    return float(scale) * (high_rate + burst)


def evaluate_shd_release_snn(h5_file, indices, snn, modulator, cfg, batch_size=32):
    snn.eval(); modulator.eval()
    correct = total = 0
    loss_sum = 0.0
    with torch.no_grad():
        for xb, yb in compressed_shd_sparse_batches(
            h5_file, indices, batch_size=batch_size, nb_steps=cfg.nb_steps,
            nb_inputs_raw=cfg.nb_inputs_raw, nb_inputs=cfg.nb_inputs, max_time=cfg.max_time,
            shuffle=False,
        ):
            logits, _, _, _ = run_shd_weight_release_modulated(xb, snn, modulator, cfg)
            loss = F.cross_entropy(logits, yb)
            pred = logits.argmax(dim=1)
            correct += int((pred == yb).sum().item())
            total += int(yb.numel())
            loss_sum += float(loss.item()) * int(yb.numel())
    return loss_sum / max(1, total), correct / max(1, total)


def train_shd_release_modulated_snn(
    root="SHD_BSS2/data",
    train_subset=1200,
    test_subset=300,
    batch_size=32,
    epochs=25,
    lr=1e-3,
    seed=41,
    cfg=None,
):
    cfg = cfg or SHDReleaseConfig()
    torch.manual_seed(seed)
    np.random.seed(seed)
    train_h5, test_h5 = open_shd_h5(root)
    train_idx = stratified_subset(train_h5["labels"], train_subset, seed=seed)
    test_idx = stratified_subset(test_h5["labels"], test_subset, seed=seed + 1)
    snn = SHDHeterogeneousSNN(cfg, seed=seed)
    modulator = SHDReleaseSiteMLP(cfg)
    optimizer = torch.optim.Adam(list(snn.parameters()) + list(modulator.parameters()), lr=lr, weight_decay=1e-5)
    history = {"loss": [], "train_acc": [], "test_acc": [], "hidden_rate": [], "event_summary": []}

    print("SHD config:", cfg)
    print("compressed channels:", cfg.nb_inputs_raw, "->", cfg.nb_inputs)
    print("ANN input dim:", modulator.input_dim, "ANN output dim:", modulator.output_dim)

    for epoch in range(1, epochs + 1):
        snn.train(); modulator.train()
        losses, correct, seen, hidden_rates = [], 0, 0, []
        first_events = []
        for batch_id, (xb, yb) in enumerate(compressed_shd_sparse_batches(
            train_h5, train_idx, batch_size=batch_size, nb_steps=cfg.nb_steps,
            nb_inputs_raw=cfg.nb_inputs_raw, nb_inputs=cfg.nb_inputs, max_time=cfg.max_time,
            shuffle=True, seed=seed + epoch,
        )):
            optimizer.zero_grad(set_to_none=True)
            events = first_events if batch_id == 0 else None
            logits, _, _, spk = run_shd_weight_release_modulated(xb, snn, modulator, cfg, modulation_events=events)
            loss = F.cross_entropy(logits, yb) + shd_spike_regularization(spk, scale=1e-3)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(snn.parameters()) + list(modulator.parameters()), 1.0)
            optimizer.step()
            losses.append(float(loss.item()))
            correct += int((logits.argmax(dim=1) == yb).sum().item())
            seen += int(yb.numel())
            hidden_rates.append(float(spk.detach().mean().item()))
        _, test_acc = evaluate_shd_release_snn(test_h5, test_idx, snn, modulator, cfg, batch_size=batch_size)
        history["loss"].append(float(np.mean(losses)))
        history["train_acc"].append(correct / max(1, seen))
        history["test_acc"].append(test_acc)
        history["hidden_rate"].append(float(np.mean(hidden_rates)))
        history["event_summary"].append(summarize_shd_release_events(first_events))
        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:03d} | loss {history['loss'][-1]:.3f} | "
                f"train {history['train_acc'][-1]:.2f} | test {test_acc:.2f} | "
                f"hidden_rate {history['hidden_rate'][-1]:.3f} | events {len(first_events)}"
            )
    train_h5.close(); test_h5.close()
    return snn, modulator, cfg, history


In [5]:
def summarize_shd_release_events(events):
    if not events:
        return {"count": 0}
    def mean_of(key, nested="abs_mean"):
        return float(np.mean([event[key][nested] for event in events]))
    return {
        "count": len(events),
        "first_step": int(events[0]["step"]),
        "last_step": int(events[-1]["step"]),
        "avg_input_spike_sum": float(np.mean([event["input_spike_sum"] for event in events])),
        "avg_hidden_spike_sum": float(np.mean([event["hidden_spike_sum"] for event in events])),
        "avg_output_mem_sum": float(np.mean([event["output_mem_sum"] for event in events])),
        "avg_amp_w1_abs": mean_of("amp_w1"),
        "avg_amp_v1_abs": mean_of("amp_v1"),
        "avg_amp_w2_abs": mean_of("amp_w2"),
        "avg_spread_h": mean_of("spread_h", "mean"),
        "avg_spread_o": mean_of("spread_o", "mean"),
        "avg_delta_w1_abs": mean_of("delta_w1"),
        "avg_delta_v1_abs": mean_of("delta_v1"),
        "avg_delta_w2_abs": mean_of("delta_w2"),
    }


def plot_shd_release_training(history):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True)
    axes[0].plot(history["loss"])
    axes[0].set_title("SHD loss")
    axes[0].set_xlabel("epoch")
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["test_acc"], label="test")
    axes[1].set_ylim(0, 1.02)
    axes[1].set_title("accuracy")
    axes[1].legend()
    axes[2].plot(history["hidden_rate"])
    axes[2].set_title("hidden spike rate")
    axes[2].set_xlabel("epoch")


def plot_shd_release_events(events):
    if not events:
        print("no modulation events recorded")
        return
    steps = np.array([event["step"] for event in events])
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), constrained_layout=True)
    axes[0].plot(steps, [event["input_spike_sum"] for event in events], label="input spike sum")
    axes[0].plot(steps, [event["hidden_spike_sum"] for event in events], label="hidden spike sum")
    axes[0].set_ylabel("window sum")
    axes[0].legend()
    axes[1].plot(steps, [event["amp_w1"]["abs_mean"] for event in events], label="|amp w1|")
    axes[1].plot(steps, [event["amp_v1"]["abs_mean"] for event in events], label="|amp v1|")
    axes[1].plot(steps, [event["amp_w2"]["abs_mean"] for event in events], label="|amp w2|")
    axes[1].set_ylabel("release amplitude")
    axes[1].legend()
    axes[2].plot(steps, [event["delta_w1"]["abs_mean"] for event in events], label="|delta w1|")
    axes[2].plot(steps, [event["delta_v1"]["abs_mean"] for event in events], label="|delta v1|")
    axes[2].plot(steps, [event["delta_w2"]["abs_mean"] for event in events], label="|delta w2|")
    axes[2].set_ylabel("weight change")
    axes[2].set_xlabel("SNN timestep")
    axes[2].legend()


def inspect_one_shd_release_sample(snn, modulator, cfg, root="SHD_BSS2/data", sample_index=0):
    _, test_h5 = open_shd_h5(root)
    try:
        events = []
        batch = next(compressed_shd_sparse_batches(
            test_h5, np.array([sample_index]), batch_size=1, nb_steps=cfg.nb_steps,
            nb_inputs_raw=cfg.nb_inputs_raw, nb_inputs=cfg.nb_inputs, max_time=cfg.max_time,
            shuffle=False,
        ))
        xb, yb = batch
        snn.eval(); modulator.eval()
        with torch.no_grad():
            logits, out_trace, mem_trace, spk_trace = run_shd_weight_release_modulated(
                xb, snn, modulator, cfg, modulation_events=events
            )
        pred = int(logits.argmax(dim=1).item())
        print("true label:", int(yb.item()), "pred:", pred)
        print("events:", summarize_shd_release_events(events))
        print("input spikes:", float(xb.sum().item()), "hidden spikes:", float(spk_trace.sum().item()))
        plot_shd_release_events(events)
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
        axes[0].imshow(xb[0].T, aspect="auto", origin="lower", cmap="magma")
        axes[0].set_title("70-channel compressed SHD input")
        axes[0].set_xlabel("timestep")
        axes[0].set_ylabel("compressed channel")
        axes[1].imshow(spk_trace[0].T, aspect="auto", origin="lower", cmap="Greys")
        axes[1].set_title("hidden spikes")
        axes[1].set_xlabel("timestep")
        axes[1].set_ylabel("hidden neuron")
        axes[2].plot(out_trace[0].cpu())
        axes[2].set_title("output membrane trace")
        axes[2].set_xlabel("timestep")
        axes[2].set_ylabel("membrane")
        return events, xb, yb, logits, out_trace, spk_trace
    finally:
        test_h5.close()



def save_shd_release_checkpoint(
    snn,
    modulator,
    cfg,
    history=None,
    events=None,
    out_dir="SHD_BSS2/checkpoints",
    name=None,
):
    """Save trained SHD SNN + ANN modulator state for later deploy/export."""
    out_dir = Path(out_dir).expanduser()
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    path = out_dir / (name or f"shd_release_snn_modulator_{stamp}.pt")
    neuron_params = {
        key: value.detach().cpu()
        for key, value in snn.neuron_params().items()
    }
    checkpoint = {
        "kind": "shd_release_modulated_snn",
        "created_at": stamp,
        "cfg": asdict(cfg),
        "snn_state_dict": {key: value.detach().cpu() for key, value in snn.state_dict().items()},
        "modulator_state_dict": {key: value.detach().cpu() for key, value in modulator.state_dict().items()},
        "neuron_params_bounded": neuron_params,
        "release_sites": {
            "hidden_sites": modulator.hidden_sites.detach().cpu(),
            "output_sites": modulator.output_sites.detach().cpu(),
            "hidden_summary": modulator.hidden_summary.detach().cpu(),
            "output_summary": modulator.output_summary.detach().cpu(),
        },
        "history": history or {},
        "events": events or [],
    }
    torch.save(checkpoint, path)
    print(f"saved SHD release checkpoint: {path}")
    return path


def load_shd_release_checkpoint(path, map_location="cpu"):
    """Reload a saved SHD release-modulated SNN checkpoint."""
    checkpoint = torch.load(Path(path).expanduser(), map_location=map_location)
    cfg = SHDReleaseConfig(**checkpoint["cfg"])
    snn = SHDHeterogeneousSNN(cfg)
    modulator = SHDReleaseSiteMLP(cfg)
    snn.load_state_dict(checkpoint["snn_state_dict"])
    modulator.load_state_dict(checkpoint["modulator_state_dict"])
    return snn, modulator, cfg, checkpoint


def print_shd_heterogeneous_param_summary(snn):
    params = snn.neuron_params()
    for name, tensor in params.items():
        data = tensor.detach().flatten()
        print(
            f"{name:8s} shape={tuple(data.shape)} mean={data.mean().item():.3f} "
            f"std={data.std(unbiased=False).item():.3f} min={data.min().item():.3f} max={data.max().item():.3f}"
        )


In [ ]:
shd_cfg = SHDReleaseConfig(
    nb_inputs=70,
    nb_inputs_raw=700,
    nb_hidden=128,
    nb_outputs=20,
    nb_steps=100,
    ann_interval=3,
    release_hidden=64,
    release_output=20,
    hidden_sizes=(256,),
    delta_scale=0.05,
)

shd_snn, shd_modulator, shd_cfg, shd_history = train_shd_release_modulated_snn(
    root="SHD_BSS2/data",
    train_subset=8156,
    test_subset=2264,
    batch_size=64,
    epochs=100,
    lr=2e-4,
    seed=41,
    cfg=shd_cfg,
)
plot_shd_release_training(shd_history)
print_shd_heterogeneous_param_summary(shd_snn)

shd_checkpoint_path = save_shd_release_checkpoint(
    shd_snn,
    shd_modulator,
    shd_cfg,
    history=shd_history,
    out_dir="SHD_BSS2/checkpoints",
)


found SHD_BSS2/data/shd_train.h5
found SHD_BSS2/data/shd_test.h5
SHD config: SHDReleaseConfig(nb_inputs=70, nb_inputs_raw=700, nb_hidden=128, nb_outputs=20, nb_steps=100, max_time=1.0, time_step=0.001, tau_syn=0.01, tau_mem=0.02, ann_interval=3, release_hidden=64, release_output=20, grid_dim='2d', spread_mode='normal', summary_width=2.0, summary_mode='uniform', delta_scale=0.05, hidden_sizes=(256,), alpha_range=(0.36787944117144233, 0.995), beta_range=(0.36787944117144233, 0.995), thr_range=(0.5, 1.5), reset_range=(-0.5, 0.5), rest_range=(-0.5, 0.5))
compressed channels: 700 -> 70
ANN input dim: 662 ANN output dim: 232
epoch 001 | loss 2.967 | train 0.06 | test 0.08 | hidden_rate 0.032 | events 34


In [ ]:
shd_events, shd_sample_x, shd_sample_y, shd_sample_logits, shd_out_trace, shd_hidden_spikes = \
    inspect_one_shd_release_sample(
        shd_snn,
        shd_modulator,
        shd_cfg,
        root="SHD_BSS2/data",
        sample_index=0,
    )

# Optional copy including recorded single-sample modulation events:
shd_checkpoint_with_events_path = save_shd_release_checkpoint(
    shd_snn,
    shd_modulator,
    shd_cfg,
    history=shd_history,
    events=shd_events,
    out_dir="SHD_BSS2/checkpoints",
    name="shd_release_snn_modulator_with_sample_events.pt",
)
